# Child Nutrition Risk Screening & Intervention Support Tool
### Colab Notebook — Data Generation + Model Training

Run all cells top to bottom. This notebook:
1. Installs dependencies
2. Generates the synthetic dataset (WHO MUAC-aligned + noise injection)
3. Trains baseline models (Logistic Regression, Random Forest) and the final XGBoost model
4. Runs SHAP explainability
5. Saves all model artifacts to `models/` (download this folder, or push to GitHub, before deploying `app.py` on Streamlit Cloud)


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost shap joblib
import os
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("Environment ready.")


## Step 1: Generate Synthetic Dataset

In [ ]:
# ============================================================================
# 01_data_generation.py
# Child Nutrition Risk Screening & Intervention Support Tool
# ----------------------------------------------------------------------------
# Generates a synthetic, WHO/ICDS-aligned dataset for children aged 6-59 months.
#
# METHODOLOGY (read this before using the data):
#   1. MUAC (Mid-Upper Arm Circumference) thresholds used here are REAL WHO
#      cutoffs for children 6-59 months:
#         MUAC < 11.5 cm        -> Severe Acute Malnutrition (SAM)
#         11.5 cm <= MUAC < 12.5 cm -> Moderate Acute Malnutrition (MAM)
#         MUAC >= 12.5 cm       -> Normal
#      Reference: WHO/UNICEF Joint Statement on MUAC screening.
#
#   2. Weight-for-age and height-for-age Z-scores are APPROXIMATED using
#      simplified age-bucketed mean/SD reference values, NOT the official
#      WHO LMS growth tables (those require the full WHO Anthro lookup
#      tables, which are not embedded here). This is an intentional,
#      documented simplification -- flagged again in the README.
#
#   3. The final risk label is NOT a pure deterministic threshold rule.
#      It is built as: MUAC-based base label -> probabilistic adjustment
#      using household/behavioural risk factors -> label noise injection.
#      This avoids trivial leakage (model just re-deriving a fixed rule)
#      and forces the model to learn genuine multi-factor signal.
#
# Output: data/child_nutrition_data.csv
# ============================================================================

import numpy as np
import pandas as pd
import os

np.random.seed(42)

N_SAMPLES = 6000
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DISTRICTS = [
    "Indore", "Bhopal", "Jabalpur", "Gwalior", "Ujjain",
    "Rewa", "Sagar", "Satna", "Ratlam", "Dewas"
]

# Approximate age-bucketed weight/height reference means & SDs (kg / cm).
# NOTE: simplified reference values for demonstration -- see README limitations.
AGE_REFERENCE = {
    # (age_min, age_max): (weight_mean, weight_sd, height_mean, height_sd)
    (6, 12):  (8.5, 1.1, 70.0, 3.0),
    (12, 24): (10.5, 1.3, 79.0, 3.5),
    (24, 36): (12.7, 1.5, 89.0, 4.0),
    (36, 48): (14.5, 1.7, 97.0, 4.2),
    (48, 60): (16.3, 1.9, 104.0, 4.5),
}


def get_reference(age_months):
    for (lo, hi), ref in AGE_REFERENCE.items():
        if lo <= age_months < hi:
            return ref
    return AGE_REFERENCE[(48, 60)]


def generate_row():
    age_months = np.random.randint(6, 59)
    gender = np.random.choice(["Male", "Female"])
    w_mean, w_sd, h_mean, h_sd = get_reference(age_months)

    # Household / behavioural risk factors
    income_bracket = np.random.choice(
        ["Low", "Lower-Middle", "Middle"], p=[0.45, 0.35, 0.20]
    )
    sanitation_access = np.random.choice(["No", "Partial", "Yes"], p=[0.30, 0.30, 0.40])
    mother_literacy = np.random.choice(["Illiterate", "Primary", "Secondary+"], p=[0.35, 0.35, 0.30])
    dietary_diversity_score = np.clip(np.random.normal(4.5, 1.8), 0, 9)  # 0-9 food groups
    immunization_status = np.random.choice(["Incomplete", "Complete"], p=[0.30, 0.70])
    district = np.random.choice(DISTRICTS)

    # Base risk pressure from socio-economic factors (drives correlated,
    # not independent, measurement generation -> realistic dataset)
    risk_pressure = 0.0
    risk_pressure += {"Low": 0.35, "Lower-Middle": 0.15, "Middle": 0.0}[income_bracket]
    risk_pressure += {"No": 0.25, "Partial": 0.10, "Yes": 0.0}[sanitation_access]
    risk_pressure += {"Illiterate": 0.20, "Primary": 0.08, "Secondary+": 0.0}[mother_literacy]
    risk_pressure += max(0, (5 - dietary_diversity_score)) * 0.05
    risk_pressure += 0.15 if immunization_status == "Incomplete" else 0.0

    # Weight/height drawn from age reference, nudged down by risk pressure
    weight = np.random.normal(w_mean - risk_pressure * 1.8, w_sd)
    height = np.random.normal(h_mean - risk_pressure * 2.5, h_sd)
    weight = max(3.0, weight)
    height = max(55.0, height)

    # MUAC correlated with weight-for-age deficit + independent noise
    weight_for_age_z = (weight - w_mean) / w_sd
    height_for_age_z = (height - h_mean) / h_sd
    muac_base = 13.5 + weight_for_age_z * 0.9
    muac = np.clip(np.random.normal(muac_base, 0.6), 8.0, 17.0)

    # ---- Base label purely from real WHO MUAC cutoffs ----
    if muac < 11.5:
        base_label = "SAM"
    elif muac < 12.5:
        base_label = "MAM"
    else:
        base_label = "Normal"

    # ---- Probabilistic adjustment using non-MUAC risk factors ----
    # Gives the model genuine multi-factor signal to learn instead of
    # just re-deriving the MUAC threshold rule.
    labels_order = ["Normal", "MAM", "SAM"]
    idx = labels_order.index(base_label)
    shift_prob = min(0.25, risk_pressure * 0.4)
    if np.random.rand() < shift_prob and idx < 2:
        idx += 1  # push toward higher risk category
    elif np.random.rand() < 0.05 and idx > 0:
        idx -= 1  # small chance of improvement (natural variance)
    final_label = labels_order[idx]

    # ---- Label noise injection (~8%) to simulate real-world
    # measurement error / imperfect field conditions ----
    if np.random.rand() < 0.08:
        final_label = np.random.choice(labels_order)

    return {
        "age_months": age_months,
        "gender": gender,
        "district": district,
        "weight_kg": round(weight, 2),
        "height_cm": round(height, 1),
        "muac_cm": round(muac, 2),
        "weight_for_age_z": round(weight_for_age_z, 2),
        "height_for_age_z": round(height_for_age_z, 2),
        "income_bracket": income_bracket,
        "sanitation_access": sanitation_access,
        "mother_literacy": mother_literacy,
        "dietary_diversity_score": round(dietary_diversity_score, 1),
        "immunization_status": immunization_status,
        "risk_category": final_label,
    }


def main():
    rows = [generate_row() for _ in range(N_SAMPLES)]
    df = pd.DataFrame(rows)

    out_path = os.path.join(OUTPUT_DIR, "child_nutrition_data.csv")
    df.to_csv(out_path, index=False)

    print(f"Generated {len(df)} rows -> {out_path}")
    print("\nClass distribution:")
    print(df["risk_category"].value_counts(normalize=True).round(3))


if __name__ == "__main__":
    main()


main()


## Step 2: Train Models & Save Artifacts

In [ ]:
# ============================================================================
# 02_model_training.py
# Trains baseline models (Logistic Regression, Random Forest) and the final
# XGBoost model on the synthetic child nutrition dataset, with:
#   - Stratified K-fold cross-validation (class imbalance is real here)
#   - Class-weighting to handle rare SAM class
#   - Per-class precision/recall/F1 reporting (NOT just accuracy)
#   - SHAP explainability
#   - Model saved in XGBoost's native JSON format (avoids sklearn/joblib
#     version-mismatch errors on Streamlit Cloud)
# ============================================================================

import json
import os

import joblib
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

DATA_PATH = "data/child_nutrition_data.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

LABEL_ORDER = ["Normal", "MAM", "SAM"]  # low -> high risk

CATEGORICAL_COLS = [
    "gender", "district", "income_bracket",
    "sanitation_access", "mother_literacy", "immunization_status",
]
NUMERIC_COLS = [
    "age_months", "weight_kg", "height_cm", "muac_cm",
    "weight_for_age_z", "height_for_age_z", "dietary_diversity_score",
]


def load_and_encode():
    df = pd.read_csv(DATA_PATH)

    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    cat_encoded = ohe.fit_transform(df[CATEGORICAL_COLS])
    cat_cols_out = ohe.get_feature_names_out(CATEGORICAL_COLS)

    X = pd.concat(
        [df[NUMERIC_COLS].reset_index(drop=True),
         pd.DataFrame(cat_encoded, columns=cat_cols_out)],
        axis=1,
    )

    le = LabelEncoder()
    le.fit(LABEL_ORDER)
    y = le.transform(df["risk_category"])

    return X, y, le, ohe


def sam_recall(y_true, y_pred, sam_idx):
    """Recall specifically on the SAM class -- the metric that matters most,
    since a missed SAM case (false negative) is far costlier than a false
    positive flag."""
    mask = y_true == sam_idx
    if mask.sum() == 0:
        return np.nan
    return (y_pred[mask] == sam_idx).mean()


def evaluate_model(model, X_test, y_test, label_encoder, name):
    y_pred = model.predict(X_test)
    sam_idx = list(label_encoder.classes_).index("SAM")

    print(f"\n===== {name} =====")
    print(classification_report(
        y_test, y_pred, target_names=label_encoder.classes_, zero_division=0
    ))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_test, y_pred))
    print(f"SAM-class recall (most important metric): "
          f"{sam_recall(y_test, y_pred, sam_idx):.3f}")

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    print(f"Macro F1: {macro_f1:.3f}")
    return macro_f1


def main():
    X, y, label_encoder, ohe = load_and_encode()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # ---------------- Baseline: Logistic Regression ----------------
    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr.fit(X_train, y_train)
    evaluate_model(lr, X_test, y_test, label_encoder, "Logistic Regression (baseline)")

    # ---------------- Baseline: Random Forest ----------------
    rf = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42
    )
    rf.fit(X_train, y_train)
    evaluate_model(rf, X_test, y_test, label_encoder, "Random Forest (baseline)")

    # ---------------- Final model: XGBoost with sample weights ----------------
    # class_weight isn't native to XGBoost multi-class; use sample_weight
    # computed from inverse class frequency to address SAM being rare.
    class_counts = np.bincount(y_train)
    class_weights = class_counts.sum() / (len(class_counts) * class_counts)
    sample_weight = np.array([class_weights[label] for label in y_train])

    # 5-fold stratified CV to sanity-check stability (rare class -> variance risk)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1_scores = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        fold_model = xgb.XGBClassifier(
            n_estimators=250, max_depth=5, learning_rate=0.08,
            objective="multi:softprob", num_class=3, eval_metric="mlogloss",
            random_state=42,
        )
        fold_model.fit(
            X_train.iloc[train_idx], y_train[train_idx],
            sample_weight=sample_weight[train_idx],
        )
        fold_pred = fold_model.predict(X_train.iloc[val_idx])
        cv_f1_scores.append(f1_score(y_train[val_idx], fold_pred, average="macro"))

    print(f"\n5-fold CV macro F1 (XGBoost): "
          f"mean={np.mean(cv_f1_scores):.3f}, std={np.std(cv_f1_scores):.3f}")

    xgb_model = xgb.XGBClassifier(
        n_estimators=250, max_depth=5, learning_rate=0.08,
        objective="multi:softprob", num_class=3, eval_metric="mlogloss",
        random_state=42,
    )
    xgb_model.fit(X_train, y_train, sample_weight=sample_weight)
    final_macro_f1 = evaluate_model(
        xgb_model, X_test, y_test, label_encoder, "XGBoost (final model)"
    )

    # Sanity check against suspiciously high accuracy (possible leakage)
    y_pred = xgb_model.predict(X_test)
    acc = (y_pred == y_test).mean()
    if acc > 0.97:
        print(
            "\nWARNING: accuracy > 97% -- re-check for label leakage before "
            "reporting this number. (Not expected with this pipeline's noise "
            "injection, but always verify.)"
        )
    else:
        print(f"\nOverall accuracy: {acc:.3f} (plausible range -- no leakage red flag)")

    # ---------------- SHAP explainability ----------------
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)
    print("\nSHAP values computed successfully "
          f"(shape info: {np.array(shap_values).shape if isinstance(shap_values, list) else shap_values.shape})")

    # ---------------- Save artifacts ----------------
    # Native XGBoost JSON format -- avoids sklearn Pipeline + joblib
    # version-mismatch errors when loading on Streamlit Community Cloud.
    xgb_model.save_model(os.path.join(MODEL_DIR, "xgb_model.json"))
    joblib.dump(ohe, os.path.join(MODEL_DIR, "onehot_encoder.pkl"))
    joblib.dump(label_encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))

    with open(os.path.join(MODEL_DIR, "feature_columns.json"), "w") as f:
        json.dump(list(X.columns), f)

    metrics = {
        "cv_macro_f1_mean": float(np.mean(cv_f1_scores)),
        "cv_macro_f1_std": float(np.std(cv_f1_scores)),
        "test_macro_f1": float(final_macro_f1),
        "test_accuracy": float(acc),
    }
    with open(os.path.join(MODEL_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"\nAll artifacts saved to '{MODEL_DIR}/'")


if __name__ == "__main__":
    main()


main()


## Step 3: Download Artifacts

Run the cell below to zip up `data/` and `models/` so you can download them and push to your GitHub repo alongside `app.py` for Streamlit Cloud deployment.


In [ ]:
!zip -r project_artifacts.zip data models
from google.colab import files
files.download("project_artifacts.zip")
